异步编程

In [ ]:
import asyncio
import nest_asyncio

# 允许嵌套事件循环
# 这个是Jupyter特殊的地方，因为Jupyter本身已经在运行事件循环了，
# 如果不启动允许事件嵌套，asyncio.run(asyncFunc(）)会出错，直接使用await调用就行
# 写异步代码：用 async def 定义函数。
# 执行异步代码：直接在单元格写 await my_func()。
# 遇到报错 cannot be called from a running event loop：删掉 asyncio.run() 改用 await，或者加上 nest_asyncio.apply()。
nest_asyncio.apply()

In [ ]:
# 标准await函数
async def main():
    print("Hello")
    await asyncio.sleep(1)
    print("Async!")

# 允许嵌套后 Jupyter 中这样写也不会报错了
asyncio.run(main())

In [ ]:
# await同步等待
async def fetch_data():
    await asyncio.sleep(2)
    return "数据"

async def main():
    print("main start")
    res = await fetch_data()
    print(res)

asyncio.run(main())

In [ ]:
# 不使用await来等待，后台并发执行
# 运行结果
"""
主程序开始   时间戳： 0
主程序继续1  时间戳： 0
后台任务开始 时间戳： 0
主程序继续2  时间戳： 5
后台任务完成 时间戳： 10
接收到task返回的数据 后台有数据   时间戳： 10
主线完成 时间戳： 10
"""


# 程序一共运行了10s，这意味着主线和后台是共用同一个时间线的
from datetime import datetime

startTime = datetime.now()


async def background_task():
    print("后台任务开始 时间戳：", (datetime.now() - startTime).seconds)
    await asyncio.sleep(10)
    print("后台任务完成 时间戳：", (datetime.now() - startTime).seconds)
    return "后台有数据"


async def main():
    print("主程序开始   时间戳：", (datetime.now() - startTime).seconds)
    # 只是注册，没有被调用
    task = asyncio.create_task(background_task())
    # 主程序继续
    print("主程序继续1  时间戳：", (datetime.now() - startTime).seconds)
    # 主程序进入等待，开始调用task
    # task调用成功后，等待10s
    # 同步
    await asyncio.sleep(5)
    # 开始调用task
    # task调用成功后，打印第一个后台任务开始
    # 同时开始task中的等待
    # 主程序5秒等待后，开始继续
    print("主程序继续2  时间戳：", (datetime.now() - startTime).seconds)
    # 等待task返回
    # task需要等待10s返回，主线中的5s和task中的5s是共用的，所以这里实际是等待了5s
    res = await task
    print(
        "接收到task返回的数据", res, "  时间戳：", (datetime.now() - startTime).seconds
    )
    # 完成
    print("主线完成 时间戳：", (datetime.now() - startTime).seconds)


asyncio.run(main())

In [ ]:
# 最后如果没有等待task返回或者调用，程序会不会直接结束
"""
主程序开始   时间戳： 707
主程序继续1  时间戳： 707
后台任务开始 时间戳： 707
主程序继续2  时间戳： 712
主线完成 时间戳： 712
后台任务完成 时间戳： 717
"""
# 答案是会的，主线任务跑完，后台还在跑
# 因为实在Jupter环境中，本来就是异步，试试一般的python环境
# 尝试下来，在py文件中，主程序会立刻结束。不会继续
# 这有区别 
# asyncNotCallInMain.py做了实验
async def background_task():
    print("后台任务开始 时间戳：", (datetime.now() - startTime).seconds)
    await asyncio.sleep(10)
    print("后台任务完成 时间戳：", (datetime.now() - startTime).seconds)
    return "后台有数据"

async def main():
    print("主程序开始   时间戳：", (datetime.now() - startTime).seconds)
    # 只是注册，没有被调用
    task = asyncio.create_task(background_task())
    # 主程序继续
    print("主程序继续1  时间戳：", (datetime.now() - startTime).seconds)
    # 主程序进入等待，开始调用task
    # task调用成功后，等待10s
    # 同步
    await asyncio.sleep(5)
    # 开始调用task
    # task调用成功后，打印第一个后台任务开始
    # 同时开始task中的等待
    # 主程序5秒等待后，开始继续
    print("主程序继续2  时间戳：", (datetime.now() - startTime).seconds)
    # 等待task返回
    # task需要等待10s返回，主线中的5s和task中的5s是共用的，所以这里实际是等待了5s
    # res = await task
    # print(
    #     "接收到task返回的数据", res, "  时间戳：", (datetime.now() - startTime).seconds
    # )
    # 完成
    print("主线完成 时间戳：", (datetime.now() - startTime).seconds)


asyncio.run(main())

主程序开始   时间戳： 1166
主程序继续1  时间戳： 1166
后台任务开始 时间戳： 1166
主程序继续2  时间戳： 1171
后台任务完成 时间戳： 1176
接收到task返回的数据 后台有数据   时间戳： 1176
主线完成 时间戳： 1176


In [ ]:
# asyncio.gather方法
# 协程处理

async def task(name, delay):
    await asyncio.sleep(delay)
    return f"任务 {name} 完成"

async def main():
    # 传入多个协程（调用时不加 await），由 gather 统一并发调度并统一 await
    results = await asyncio.gather(
        task("A", 2),
        task("B", 1),
        task("C", 1.5)
    )
    print(results)  # 总耗时只有 2 秒（最长的那项），而不是 2+1+1.5 秒

asyncio.run(main())

['任务 A 完成', '任务 B 完成', '任务 C 完成']
